In [3]:
import requests
import yfinance as yf
import json
import time
import os

API_KEY = "PFNCG4FDSWHLTRV5"
QUARTER_MONTH = {"Q1":"02", "Q2":"05", "Q3":"08", "Q4":"11"}
os.makedirs("data/transcripts", exist_ok=True)

tickers = [
    "AAPL", "MSFT", "GOOGL", "NVDA", "META",
    "JPM", "BAC", "GS", "V", "MA",
    "JNJ", "PFE", "UNH", "ABBV", "MRK",
    "AMZN", "TSLA", "MCD", "NKE", "SBUX",
    "XOM", "CVX", "HD", "HON", "UPS"
]

quarters = ["2023Q4", "2024Q1", "2024Q2"]


In [4]:
MAX_CALLS = 25
calls_made = 0

for ticker in tickers:
    if calls_made >= MAX_CALLS:
        print("Hit daily limit, run again tomorrow!")
        break
    for quarter in quarters:
        if calls_made >= MAX_CALLS:
            break
        
        filename = f"data/transcripts/{ticker}_{quarter}.json"
        if os.path.exists(filename):
            print(f"Already have {ticker} {quarter}, skipping")
            continue
        
        # fetch transcript
        params = {"function":"EARNINGS_CALL_TRANSCRIPT","symbol":ticker,"quarter":quarter,"apikey":API_KEY}
        transcript_data = requests.get("https://www.alphavantage.co/query", params=params).json()
        
        # fetch return
        year, q = quarter[:4], quarter[4:]
        date = f"{year}-{QUARTER_MONTH[q]}-01"
        hist = yf.Ticker(ticker).history(start=date, period="10d")
        return_pct = (float(hist["Close"].iloc[3]) - float(hist["Close"].iloc[0])) / float(hist["Close"].iloc[0]) * 100
        
        # save
        transcript_data["return_pct"] = return_pct
        transcript_data["earnings_date"] = date
        transcript_data["symbol"] = ticker
        transcript_data["quarter"] = quarter
        with open(filename, "w") as f:
            json.dump(transcript_data, f)
        
        print(f"✓ {ticker} {quarter}: {return_pct:.2f}%")
        calls_made += 1
        time.sleep(13)

print(f"Done! Collected {calls_made} transcripts today.")

✓ AAPL 2023Q4: 3.02%
Already have AAPL 2024Q1, skipping
✓ AAPL 2024Q2: 7.33%
✓ MSFT 2023Q4: 3.02%
✓ MSFT 2024Q1: 0.42%
✓ MSFT 2024Q2: 4.71%
✓ GOOGL 2023Q4: 3.01%
✓ GOOGL 2024Q1: 2.08%
✓ GOOGL 2024Q2: 2.59%
✓ NVDA 2023Q4: 8.09%
✓ NVDA 2024Q1: 8.24%
✓ NVDA 2024Q2: 10.96%
✓ META 2023Q4: 1.27%
✓ META 2024Q1: 15.18%
✓ META 2024Q2: 6.03%
✓ JPM 2023Q4: 3.70%
✓ JPM 2024Q1: 0.79%
✓ JPM 2024Q2: 0.07%
✓ BAC 2023Q4: 7.31%
✓ BAC 2024Q1: -1.52%
✓ BAC 2024Q2: 2.00%
✓ GS 2023Q4: 5.45%
✓ GS 2024Q1: 0.30%
✓ GS 2024Q2: 3.92%
✓ V 2023Q4: 2.06%
✓ V 2024Q1: -0.10%
Hit daily limit, run again tomorrow!
Done! Collected 25 transcripts today.


In [6]:
from pathlib import Path
import json

data_dir = Path("/gpfs/home/smaccall/earnings-call-prediction/notebooks/data/transcripts")

missing = []
for path in data_dir.glob("*.json"):
    with open(path) as f:
        d = json.load(f)
    if "return_pct" not in d:
        missing.append(path.name)
print(f"{len(missing)} files missing return_pct:")
print(missing)
missing = []
for path in data_dir.glob("*.json"):
    with open(path) as f:
        d = json.load(f)
    if "return_pct" not in d:
        missing.append(path.name)
print(f"{len(missing)} files missing return_pct:")
print(missing)

0 files missing return_pct:
[]
0 files missing return_pct:
[]
